[Lab README](README.md)

# Lab 4.1: The write, and the rule that stops it

Lab 3 blocked a 15-guest booking with `MaxGuestsHook`. That worked. The hook
fires before the tool runs, the model receives a cancellation instead of a
result, and no amount of rephrasing gets around it. As a demonstration that a
guardrail can sit outside the model's reach, it is exactly right.

The number is the problem. `10` is a Python literal inside a hook class, in one
notebook, next to one agent. The moment a second caller needs the same limit,
there is nothing to read it from, and the two copies start drifting the day
someone changes one of them. A business rule is connected data: it belongs where
the hotels and reservations already live, enforced inside the same boundary as
the write it governs.

This lab moves it there. The limit comes out of a `Rule` node in your graph, the
reservation command reads it inside the same transaction as the write, and the
same 15-guest request gets rejected without anything being written.

## Who owns what

| Neo4j owns | AWS owns |
|---|---|
| The maximum-guests rule, as a `Rule` node your command reads | Amazon Bedrock reasons over the retrieved evidence |
| The idempotent `ReservationRequest` write and its `FOR_HOTEL` link | |
| The uniqueness constraints that make a retry safe | |

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the line below first.
# !pip install -r requirements.txt

print("Environment ready")

## 1. Connect, and confirm the rule is in the graph

The reservation command needs three things from your graph: the fixture hotel
IDs, three uniqueness constraints, and the `Rule` node. Lab 1 seeded all of
them. The cell below applies them again, which is idempotent, then reports
anything still missing.

In [ ]:
import json
import os
import uuid
from datetime import date, timedelta

import boto3
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Safe at the top of the cell: bedrock_providers opens no client and reads
# no required variable at import. The workshop imports further down are
# inside the guard because graph_connection raises without NEO4J_PASSWORD.
from workshop.bedrock_providers import default_model_id

load_dotenv()

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
AGENT_READY = NEO4J_READY and BEDROCK_READY
RUNNER_SKIP_WRITES = os.getenv("WORKSHOP_RUNNER") == "1"
WRITE_READY = NEO4J_READY and not RUNNER_SKIP_WRITES
AGENT_WRITE_READY = AGENT_READY and not RUNNER_SKIP_WRITES

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
# One definition, in workshop/src/workshop/bedrock_providers.py. A MODEL_ID
# in the environment still overrides it.
MODEL_ID = default_model_id()

if not NEO4J_READY:
    print("Neo4j is not configured; the live cells below will skip.")
if not BEDROCK_READY:
    print("AWS credentials are not configured; the agent cells below will skip.")

if NEO4J_READY:
    from workshop.contracts import MAX_GUESTS, MAX_GUESTS_RULE_ID, OVER_LIMIT_GUESTS
    from workshop.graph_setup import (
        HERO_NAME,
        HERO_SOURCE,
        RULE_QUERY,
        apply_lab4_fixtures,
        load_manifest,
        readiness_problems,
    )
    from workshop.hybrid_retrieval import Neo4jConfig
    from workshop.reservation_command import create_reservation_request

    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    # notifications_min_severity="OFF" keeps Neo4j's 01N51 and 01N52 planner
    # notices out of the output. The shared package sets it on its drivers too.
    driver = GraphDatabase.driver(
        config.uri,
        auth=(config.username, config.password),
        notifications_min_severity="OFF",
    )
    driver.verify_connectivity()

    problems = apply_lab4_fixtures(driver, config.database, manifest)
    if not problems:
        problems = readiness_problems(driver, config.database, manifest)
    if problems:
        raise RuntimeError(
            "The graph is not ready: "
            + "; ".join(problems)
            + "\nRe-run 01-graph-build/1.1_build_graph.ipynb."
        )
    print("Graph ready: fixtures applied, constraints present, rule seeded.")

### Where the limit lives now

Read it before writing anything. This is the whole argument of the lab in one
cell: the number is not in this notebook, not in a prompt, and not in a hook
class. It is a property on a node, with a rejection message and an enable flag
next to it, and any caller that can reach the graph can read the same value.

The rule id prints as `demo-06-maximum-guests`, and the three uniqueness
constraints are named `demo06_fixture_hotel_id`,
`demo06_reservation_request_id`, and `demo06_rule_id`. Those five `demo06`
spellings are frozen on purpose. They are the real names of objects inside a
graph, so renaming them to match the lab numbering would leave every graph
already built carrying the old names and the new code looking for the new ones.
Identifiers that only name Python, such as file and class names, were renamed;
identifiers that name graph state were not.

In [ ]:
if not NEO4J_READY:
    print("Skipping: Neo4j is not configured.")
else:
    with driver.session(database=config.database) as session:
        rule = session.run(RULE_QUERY, rule_id=MAX_GUESTS_RULE_ID).single()

    print(f"Rule node:  {MAX_GUESTS_RULE_ID}")
    print(f"  max_guests:        {rule['max_guests']}")
    print(f"  enabled:           {rule['enabled']}")
    print(f"  rejection_message: {rule['rejection_message']}")
    print(f"  steering_message:  {rule['steering_message']}")

    # The Python constant and the graph must agree. When they do not, the graph
    # is the one that decides, because it is what the command reads.
    assert rule["max_guests"] == MAX_GUESTS
    print(f"\nPython constant MAX_GUESTS={MAX_GUESTS} agrees with the graph.")

## 2. Register the write on `hotel_agent`

Lab 3 finished with `hotel_agent`: the hybrid retriever behind a `@tool`, a
simulated `book_room` tool, and a guest limit enforced by a hook. Rebuild it
here with four changes.

1. The write tool `create_reservation_request` joins the toolset.
2. `MaxGuestsHook` comes off, because the rule it enforced now lives in the
   graph and the command reads it.
3. The system prompt gains one instruction: pass the caller's identifiers
   through unchanged, and report the command's response rather than composing
   one.
4. `book_room` comes off, and this is the change worth pausing on. It returned
   `f"SUCCESS: Booked {hotel} for {guests} guests"` no matter what it was
   handed. Every booking it ever reported was a string, and a hook was the only
   thing standing between the model and an unlimited supply of them. Lab 3
   needed a tool that did nothing so the hook had something to cancel. From here
   the tool writes to your graph, so a `SUCCESS` the graph did not agree to is
   no longer available to the agent.

The tool signature is exactly the five fields of the reservation contract.
`request_id` is created by the caller, not the model, and reusing it is what
makes a retry safe.

In [ ]:
if not AGENT_READY:
    print("Skipping: needs both Neo4j and AWS credentials.")
else:
    from strands import Agent, tool
    from strands.models import BedrockModel

    from workshop.hybrid_retrieval import GROUNDING_INSTRUCTIONS, search_hotel_knowledge

    @tool
    def search_hotel_knowledge_tool(query: str) -> str:
        """Look up amenities, ratings, and policies for a specific named hotel."""
        return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

    @tool
    def create_reservation_request_tool(
        request_id: str,
        hotel_id: str,
        check_in: str,
        check_out: str,
        guests: int,
    ) -> str:
        """Create a reservation request. Reuse request_id when retrying."""
        # The tool opens and closes its own driver rather than closing over the
        # notebook's, which the cells around it reuse. In Lab 5 this same tool
        # body runs inside a Lambda holding a different, write-scoped Neo4j
        # credential, so it has to own its connection to be liftable there
        # unchanged.
        write_driver = GraphDatabase.driver(
            config.uri,
            auth=(config.username, config.password),
            notifications_min_severity="OFF",
        )
        try:
            response = create_reservation_request(
                {
                    "request_id": request_id,
                    "hotel_id": hotel_id,
                    "check_in": check_in,
                    "check_out": check_out,
                    "guests": guests,
                },
                driver=write_driver,
                database=config.database,
            )
        finally:
            write_driver.close()
        return json.dumps(response, ensure_ascii=False)

    hotel_agent = Agent(
        name="hotel_agent",
        model=BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION),
        tools=[search_hotel_knowledge_tool, create_reservation_request_tool],
        system_prompt=(
            "You are a hotel assistant. Look up hotels with "
            "search_hotel_knowledge_tool and take bookings only with "
            "create_reservation_request_tool. Pass through the request_id, "
            "hotel_id, and dates the user gives you exactly as written. Report "
            "the command's response as it comes back; never claim a booking "
            "succeeded unless the response says accepted. When a command rejects a request, report only facts present in its response and do not add remedies or alternatives.\n\n"
            + GROUNDING_INSTRUCTIONS
        ),
    )

    WRITE_TOOL = "create_reservation_request_tool"

    def tool_calls(tool_name: str) -> int:
        """Times the named tool has run on hotel_agent since it was built.

        Strands accumulates tool metrics on the agent rather than resetting
        them per turn, so the cells below compare a count taken before a turn
        against the count after it. Without that, a model that answered from
        the prompt and never called the tool would still leave the notebook
        green.
        """
        metric = hotel_agent.event_loop_metrics.tool_metrics.get(tool_name)
        return metric.call_count if metric else 0

    print("hotel_agent now carries the retriever and the reservation write.")

### The limit is nowhere in the prompt

That claim carries the rest of the lab, so read the prompt rather than trusting
the claim. `GROUNDING_INSTRUCTIONS` arrives by import and is concatenated on the
end, which makes it the easiest place for a stray number to hide.

Print the assembled prompt and check it character by character. No digit appears
anywhere in it, so there is no `10`, no `15`, and no threshold of any kind for
the model to be argued out of.

In [ ]:
if not AGENT_READY:
    print("Skipping: needs both Neo4j and AWS credentials.")
else:
    print("=== hotel_agent.system_prompt, in full ===\n")
    print(hotel_agent.system_prompt)

    digits = sorted({c for c in hotel_agent.system_prompt if c.isdigit()})
    print("\n=== digits found in the prompt ===")
    print(digits or "none")

    assert not digits, (
        f"The system prompt now contains {digits}. Every claim below rests on "
        "the model having no number to reason from, so a numeric threshold in "
        "the prompt breaks the lab rather than just weakening it."
    )

## 3. A 15-guest request is rejected, and nothing is written

The dates are computed from today, so this example never expires. Three
`request_id` values are created here, one per case, because the three cases make
different points. The over-limit id shows that a rejected request leaves nothing
behind. The agent booking id, used in section 4, shows the agent taking a real
action that lands in the graph. The booking id, also section 4, shows that the
same id delivered twice produces one node.

In [ ]:
if not NEO4J_READY:
    print("Skipping: Neo4j is not configured.")
else:
    hero_id = manifest.hotels[HERO_SOURCE]
    check_in = (date.today() + timedelta(days=30)).isoformat()
    check_out = (date.today() + timedelta(days=32)).isoformat()
    OVER_LIMIT_REQUEST_ID = str(uuid.uuid4())
    AGENT_BOOKING_REQUEST_ID = str(uuid.uuid4())
    BOOKING_REQUEST_ID = str(uuid.uuid4())

    print(f"Hero hotel_id (from fixture manifest):    {hero_id}")
    print(f"Over-limit request_id (never written):    {OVER_LIMIT_REQUEST_ID}")
    print(f"Agent booking request_id (written once):  {AGENT_BOOKING_REQUEST_ID}")
    print(f"Booking request_id (reused on retries):   {BOOKING_REQUEST_ID}")
    print(f"Stay: {check_in} to {check_out}")

### That id is the one retrieval hands back

`hero_id` came out of the committed fixture manifest, which is convenient and
proves nothing on its own. The word grounded means the id the write path accepts
is the same id the retriever returns from evidence in the graph, so run the Lab 2
retriever once and compare the two.

The comparison is between the top result's `hotel_id` and `hero_id`. A display
name is not compared, because a display name is exactly what the command refuses
to accept.

In [ ]:
if not AGENT_READY:
    print("Skipping: needs both Neo4j and AWS credentials.")
else:
    hero_question = f"What amenities and guest rating does {HERO_NAME} have?"
    evidence = search_hotel_knowledge(hero_question)
    top = evidence[0]

    print(f"Question:  {hero_question}\n")
    print(f"Top result hotel_name: {top['hotel_name']}")
    print(f"Top result hotel_id:   {top['hotel_id']}")
    print(f"Fixture manifest id:   {hero_id}")
    print(f"Combined hybrid score: {top['combined_score']:.4f}")

    assert top["hotel_id"] == hero_id, (
        "The retriever's top hotel_id does not match the fixture manifest, so "
        "the id the write path is about to accept is not the id retrieval "
        "returns. Re-run Lab 1 before reading anything below as grounded."
    )
    print("\nRetrieval and the write path name the same hotel by the same id.")

In [ ]:
if not AGENT_WRITE_READY:
    print(
        "Skipping write scenario: the runner blocks writes or live "
        "Neo4j and AWS configuration is missing."
    )
else:
    before = tool_calls(WRITE_TOOL)
    # Strands prints the streamed response itself, through the default
    # PrintingCallbackHandler. Wrapping this call in print() would show the
    # whole answer a second time.
    hotel_agent(
        f"Book {OVER_LIMIT_GUESTS} guests into hotel_id {hero_id} from "
        f"{check_in} to {check_out}. Use request_id {OVER_LIMIT_REQUEST_ID}."
    )

    after = tool_calls(WRITE_TOOL)
    print(f"\n{WRITE_TOOL} calls this turn: {after - before}")
    assert after > before, (
        "The agent answered without calling the write tool. A refusal the "
        "model composed on its own looks the same in the transcript as a "
        "refusal the graph enforced, and only the second one is the point of "
        "this lab."
    )

The agent tried, and the tool-call count above says so. The command refused. The
prompt you printed a moment ago held no number at all, so the model had no limit
to be talked out of, and the rejection came back as a structured response rather
than as an apology it composed.

Here is that response on its own, called directly, so you can read the fields the
agent was working from. The same payload goes in twice, which brings in the flag
section 4 is built around: `duplicate` is the command's way of saying "I have
this request already, and I am returning what I stored rather than writing
again." It can only ever be true against a request that was stored.

That is what makes the second delivery worth watching here. It comes back with
`duplicate` still false, because the rejection stored nothing for a second
delivery to find.

In [ ]:
if not WRITE_READY:
    print("Skipping rule rejection write: runner safety or no Neo4j.")
else:
    over_limit_payload = {
        "request_id": OVER_LIMIT_REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": OVER_LIMIT_GUESTS,
    }
    rejected = create_reservation_request(
        over_limit_payload, driver=driver, database=config.database
    )
    rejected_again = create_reservation_request(
        over_limit_payload, driver=driver, database=config.database
    )
    print("Rejection, in full:")
    print(json.dumps(rejected, indent=2))
    print("\nSame request_id re-delivered:")
    print(f"  status:    {rejected_again['status']}")
    print(f"  duplicate: {rejected_again['duplicate']}")
    print("\nduplicate is false because the first delivery wrote nothing to find.")

    assert rejected["status"] == "rejected"
    assert rejected["reason_code"] == "max_guests_exceeded"
    assert rejected_again["duplicate"] is False

## 4. The agent books a stay, and the row is in your graph

Every agent turn so far has been a refusal, and a lab that only ever shows an
agent being blocked has argued half its case. A guardrail is worth having
because the thing it guards is allowed to happen. So here the agent gets a
request the rule permits, and it goes through.

Same hotel, same dates, a party size under the limit, and the agent booking
`request_id` from section 3. The agent calls the same tool that was refused a
moment ago, against the same command, reading the same rule. The next cell then
goes to the graph and reads the row back, because the agent saying it booked and
the graph holding a booking are two different claims.

In [ ]:
AGENT_BOOKING_GUESTS = 4

if not AGENT_WRITE_READY:
    print(
        "Skipping write scenario: the runner blocks writes or live "
        "Neo4j and AWS configuration is missing."
    )
else:
    before = tool_calls(WRITE_TOOL)
    hotel_agent(
        f"Book {AGENT_BOOKING_GUESTS} guests into hotel_id {hero_id} from "
        f"{check_in} to {check_out}. Use request_id {AGENT_BOOKING_REQUEST_ID}."
    )

    after = tool_calls(WRITE_TOOL)
    print(f"\n{WRITE_TOOL} calls this turn: {after - before}")
    assert after > before, (
        "The agent reported a booking without calling the write tool. That is "
        "the Lab 3 failure in a new costume: a SUCCESS string with nothing "
        "behind it."
    )

In [ ]:
# Section 6 reads rows with this same query, so it is defined once, here, and
# outside the guard below.
REQUEST_QUERY = (
    "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
    "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
    "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
    "toString(r.created_at) AS created_at"
)

if not AGENT_WRITE_READY:
    print("Skipping agent write inspection: runner safety or missing config.")
else:
    with driver.session(database=config.database) as session:
        agent_rows = [
            dict(record)
            for record in session.run(REQUEST_QUERY, rid=AGENT_BOOKING_REQUEST_ID)
        ]
    for row in agent_rows:
        print(row)

    assert len(agent_rows) == 1, (
        f"expected one row for the agent's booking, found {len(agent_rows)}"
    )
    assert agent_rows[0]["status"] == "accepted"
    assert agent_rows[0]["guests"] == AGENT_BOOKING_GUESTS
    assert agent_rows[0]["hotel_id"] == hero_id
    print("\nThe agent's booking is a row in your graph, on the id you gave it.")

### The same request delivered twice

The booking `request_id` now, called directly so both responses are in hand at
once, and a party size at the limit rather than under it. The command creates
one `ReservationRequest` linked to the hero hotel by `FOR_HOTEL`. Re-delivering
the identical request returns the existing record with `duplicate=true` and
creates no second node, so a retried or replayed reservation is safe.

The proof is in the two responses side by side. Apart from the `duplicate` flag
and the message that follows from it, they are the same record, down to the same
`created_at`, so the second delivery read the first one back rather than writing
its own.

That guarantee is a uniqueness constraint in the graph, not a check the caller
remembered to write. It holds whether the retry comes from this notebook, the
agent, or a Lambda that timed out and was retried by its caller.

One note on re-running. The cell below asserts that its first delivery is a
first delivery, so running it a second time on the same `request_id` fails on
`accepted["duplicate"]`, correctly. Re-run the section 3 cell that mints the
three ids first, then this one.

In [ ]:
if not WRITE_READY:
    print("Skipping valid write: runner safety or no Neo4j.")
else:
    valid_payload = {
        "request_id": BOOKING_REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": MAX_GUESTS,
    }
    accepted = create_reservation_request(
        valid_payload, driver=driver, database=config.database
    )
    replay = create_reservation_request(
        valid_payload, driver=driver, database=config.database
    )
    print("First delivery:")
    print(json.dumps(accepted, indent=2))
    print("\nSame request_id re-delivered:")
    print(json.dumps(replay, indent=2))

    # The two responses describe one record. Strip the duplicate flag and the
    # message it drives, and what is left has to be identical, character for
    # character, including the Neo4j-generated created_at.
    record = {"status", "request_id", "hotel_id", "created_at"}
    first_record = json.dumps({k: accepted[k] for k in sorted(record)})
    second_record = json.dumps({k: replay[k] for k in sorted(record)})
    print(f"\nSame record both times: {first_record == second_record}")
    print(f"  {first_record}")

    assert accepted["status"] == "accepted" and not accepted["duplicate"]
    assert replay["duplicate"] is True
    assert first_record == second_record

## 5. A hotel that does not exist is rejected

The rule is one of two things the command checks. The other is identity: a
`hotel_id` that matches no `Hotel` node cannot be booked, and the command says
so with a reason code rather than creating an orphan request.

This is why grounded retrieval returns an opaque `hotel_id` and not a display
name. A name a model half-remembers will fail this check. A name it invents
outright would too.

In [ ]:
if not AGENT_WRITE_READY:
    print(
        "Skipping write scenario: the runner blocks writes or live "
        "Neo4j and AWS configuration is missing."
    )
else:
    before = tool_calls(WRITE_TOOL)
    # As above, Strands prints the streamed response; no print() wrapper here.
    hotel_agent(
        f"Book 2 guests into hotel_id hotel-does-not-exist from "
        f"{check_in} to {check_out}. Use request_id {uuid.uuid4()}."
    )

    after = tool_calls(WRITE_TOOL)
    print(f"\n{WRITE_TOOL} calls this turn: {after - before}")
    assert after > before, (
        "The agent decided the hotel was fake without asking the command. "
        "Identity is the command's check to make, not the model's guess to "
        "make, and this turn is only evidence when the tool actually ran."
    )

The agent passed the invented id straight through, as instructed, and the
command was the thing that said no. That division is deliberate: the model is
not asked to know which hotels exist, only to hand the id along and report what
comes back.

Here is the same rejection called directly, so the `reason_code` is readable
next to the `max_guests_exceeded` one from section 3. Two rejections, two codes,
one command, and neither of them written.

In [ ]:
if not WRITE_READY:
    print("Skipping unknown-hotel write: runner safety or no Neo4j.")
else:
    unknown_payload = {
        "request_id": str(uuid.uuid4()),
        "hotel_id": "hotel-does-not-exist",
        "check_in": check_in,
        "check_out": check_out,
        "guests": 2,
    }
    unknown = create_reservation_request(
        unknown_payload, driver=driver, database=config.database
    )
    print(json.dumps(unknown, indent=2))
    assert unknown["status"] == "rejected"
    assert unknown["reason_code"] == "unknown_hotel"

## 6. Inspect the reservation in your graph

Three ids, three different expectations, all read back from the graph rather
than from a response the command handed you.

The booking id was delivered twice and must show exactly one row. The agent
booking id was written once, by the agent, and must also show exactly one row.
The over-limit id was delivered twice here and once more by the agent, and must
show no row at all, which is the claim section 3 made and this cell finally
checks.

In [ ]:
if not WRITE_READY:
    print("Skipping write inspection: runner safety or no Neo4j.")
else:
    with driver.session(database=config.database) as session:
        booking_rows = [
            dict(record)
            for record in session.run(REQUEST_QUERY, rid=BOOKING_REQUEST_ID)
        ]
        over_limit_rows = [
            dict(record)
            for record in session.run(REQUEST_QUERY, rid=OVER_LIMIT_REQUEST_ID)
        ]

    print(f"Booking request_id rows:    {len(booking_rows)}")
    for row in booking_rows:
        print(f"  {row}")
    print(f"Over-limit request_id rows: {len(over_limit_rows)}")
    for row in over_limit_rows:
        print(f"  {row}")

    assert len(booking_rows) == 1, (
        f"expected exactly one request, found {len(booking_rows)}"
    )
    assert not over_limit_rows, (
        f"the rejected request_id left {len(over_limit_rows)} rows behind, so "
        "the rule rejected the request and wrote it anyway"
    )

    if AGENT_READY:
        with driver.session(database=config.database) as session:
            agent_booking_rows = [
                dict(record)
                for record in session.run(
                    REQUEST_QUERY, rid=AGENT_BOOKING_REQUEST_ID
                )
            ]
        print(f"Agent booking request_id rows: {len(agent_booking_rows)}")
        for row in agent_booking_rows:
            print(f"  {row}")
        assert len(agent_booking_rows) == 1, (
            f"expected one agent booking, found {len(agent_booking_rows)}"
        )

## What you built

| | Where it lives | What it means |
|---|---|---|
| The guest limit | A `Rule` node in Neo4j | One value, readable by every caller, changed in one place |
| The rejection | Inside the write transaction | An over-limit request cannot be written, not even by a retry |
| Hotel identity | A `hotel_id` from grounded retrieval | An invented or half-remembered name fails the check |
| Retry safety | A uniqueness constraint | The second delivery returns the first record, not a second node |
| The booking the agent made | A row you read back from the graph | The agent is trusted with the action, and the graph is what confirms it happened |

Everything above ran on your own Aura instance and Amazon Bedrock, and created
no AWS resources.

**Next:** Lab 5 takes this same retrieval tool and this same reservation command
and runs them as a managed service on Amazon Bedrock AgentCore. The contracts do
not change. The trust boundary does.